# Clasificador de color – NTP 900.058 (7 colores)
Entrena **YOLO11n-cls** con el dataset de `data basurita` (creado con `dataset_ntp900058_splits.ipynb`).

En el detector trabaja en dos pasos: **YOLO-World ubica el objeto** y **este clasificador decide el color**
mirando el recorte. Así distingue el material (plástico vs. papel) con miles de fotos reales.

Entrenamiento en 2 etapas: **base congelada** y luego **ajuste fino**. Tarda ~20–40 min con GPU T4.

**Antes de empezar:** `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4`.

## 1. Instalar y verificar la GPU

In [ ]:
!pip install -q "ultralytics>=8.3,<8.4"
import torch
print("PyTorch:", torch.__version__, "| GPU:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 2. Conectar Drive y configuración

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/data basurita"
ZIP = f"{DRIVE_DIR}/dataset_ntp900058.zip"
DATASET = "/content/dataset_cls"            # copia local (mucho mas rapida que leer del Drive)
RUNS = f"{DRIVE_DIR}/runs_clasificador"     # resultados en el Drive, por si Colab se desconecta
SEMILLA = 42

# Carpetas del dataset -> nombres de clase del detector
NOMBRES = {
    "Azul": "azul_papel_carton",
    "Amarillo": "amarillo_metales",
    "Blanco": "blanco_plasticos",
    "Gris": "gris_vidrio",
    "Marron": "marron_organicos",
    "Rojo": "rojo_peligrosos",
    "Negro": "negro_generales",
}

## 3. Copiar el dataset desde el Drive
Usa el `.zip` (rápido). Si no existe, copia la carpeta `splits`.
Las carpetas se renombran a los nombres de clase del detector.

In [ ]:
import os, shutil, zipfile

shutil.rmtree(DATASET, ignore_errors=True)
if os.path.exists(ZIP):
    zipfile.ZipFile(ZIP).extractall(DATASET)
else:
    shutil.copytree(f"{DRIVE_DIR}/splits", DATASET)

for split in ("train", "val", "test"):
    for corto, largo in NOMBRES.items():
        origen = f"{DATASET}/{split}/{corto}"
        if os.path.isdir(origen):
            os.rename(origen, f"{DATASET}/{split}/{largo}")

print(f"{'clase':<20}{'train':>8}{'val':>8}{'test':>8}")
for c in NOMBRES.values():
    n = [len(os.listdir(f"{DATASET}/{s}/{c}")) if os.path.isdir(f"{DATASET}/{s}/{c}") else 0
         for s in ("train", "val", "test")]
    aviso = "   <-- AVISO: vacio" if 0 in n else ""
    print(f"{c:<20}{n[0]:>8}{n[1]:>8}{n[2]:>8}{aviso}")

## 4. Etapa 1 – base congelada
Parte de `yolo11n-cls.pt` (preentrenado en ImageNet) y **congela las 10 capas de la base**:
solo se entrena la capa final que decide entre los 7 colores.

In [ ]:
from ultralytics import YOLO

modelo = YOLO("yolo11n-cls.pt")
modelo.train(
    data=DATASET,
    epochs=25,
    imgsz=224,
    batch=64,
    freeze=10,
    patience=8,
    seed=SEMILLA,
    project=RUNS,
    name="etapa1_congelada",
    exist_ok=True,
)

## 5. Etapa 2 – ajuste fino (todo descongelado, aprendizaje muy bajo)
La base se adapta a las texturas de los residuos (plástico, papel, vidrio…) sin olvidar lo general.

In [ ]:
modelo = YOLO(f"{RUNS}/etapa1_congelada/weights/best.pt")
modelo.train(
    data=DATASET,
    epochs=15,
    imgsz=224,
    batch=64,
    optimizer="SGD",   # con optimizer="auto" se ignora lr0
    lr0=0.001,
    lrf=0.1,
    warmup_epochs=0,
    patience=6,
    seed=SEMILLA,
    project=RUNS,
    name="etapa2_ajuste_fino",
    exist_ok=True,
)

In [ ]:
# (Solo si Colab se desconecto) Reanudar la etapa que se corto:
# YOLO(f"{RUNS}/etapa1_congelada/weights/last.pt").train(resume=True)
# YOLO(f"{RUNS}/etapa2_ajuste_fino/weights/last.pt").train(resume=True)

## 6. Elegir el mejor (con **val**) y evaluarlo una sola vez en **test**
> El dataset se dividió foto por foto al azar. Si hay fotos casi iguales en train y test,
> la precisión de test puede salir algo más alta que en la cámara real.

In [ ]:
from IPython.display import Image as IPImage, display

candidatos = {e: f"{RUNS}/{e}/weights/best.pt" for e in ("etapa1_congelada", "etapa2_ajuste_fino")}
puntaje = {}
for etapa, ruta in candidatos.items():
    if os.path.exists(ruta):
        m = YOLO(ruta).val(data=DATASET, split="val", plots=False, verbose=False)
        puntaje[etapa] = m.top1
        print(f"{etapa:<20} val precision top-1: {m.top1:.3f}")
GANADOR = max(puntaje, key=puntaje.get)
print("\nMejor:", GANADOR)

m = YOLO(candidatos[GANADOR]).val(data=DATASET, split="test", project=RUNS, name="evaluacion", exist_ok=True)
print(f"\ntest: precision top-1 {m.top1:.3f} | top-5 {m.top5:.3f}")
matriz = m.confusion_matrix.matrix  # filas = prediccion, columnas = real
nombres = list(YOLO(candidatos[GANADOR]).names.values())
for i, c in enumerate(nombres):
    total = matriz[:, i].sum()
    if total:
        print(f"  {c:<20} acierto {matriz[i, i] / total:.1%}  ({int(total)} fotos)")

In [ ]:
ruta = f"{RUNS}/evaluacion/confusion_matrix_normalized.png"
if os.path.exists(ruta):
    display(IPImage(ruta, width=700))

## 7. Probar con una foto tuya

In [ ]:
from google.colab import files
mejor = YOLO(candidatos[GANADOR])
for nombre in files.upload():
    r = mejor.predict(nombre, imgsz=224, verbose=False)[0]
    top = r.probs.top5[:3]
    print(nombre, "->", ", ".join(f"{r.names[i]} {float(r.probs.data[i]):.0%}" for i in top))
    display(IPImage(nombre, width=300))

## 8. Guardar y descargar `clasificador.pt`
Cópialo a `modelos/clasificador.pt` en tu Mac. El detector lo usa automáticamente junto con YOLO-World.

In [ ]:
os.makedirs(f"{DRIVE_DIR}/modelos", exist_ok=True)
shutil.copy(candidatos[GANADOR], f"{DRIVE_DIR}/modelos/clasificador.pt")
shutil.copy(candidatos[GANADOR], "/content/clasificador.pt")
print("Copia en el Drive:", f"{DRIVE_DIR}/modelos/clasificador.pt")
files.download("/content/clasificador.pt")